## Nifty 50 Options Trading Strategy Bot

This notebook implements a Nifty 50 options trading strategy with dynamic profit locking, re-entry rules, and safety features.
Supports both **Paper Trading Mode** (zero API keys needed) and **Live Zerodha Kite Connect API Mode**.


In [ ]:
# Install dependencies
!pip install kiteconnect pandas --quiet


In [ ]:
# -*- coding: utf-8 -*-
"""Historydownload.ipynb

Automatically generated by Colab / AlgoBeat Engine.

## Nifty 50 Options Trading Strategy Bot

This script implements a Nifty 50 options trading strategy with dynamic profit locking,
re-entry rules, safety features, paper trading simulation, and live Zerodha Kite Connect integration.
"""

import os
import sys
import logging
import datetime
import time
import math
import pandas as pd
from collections import defaultdict

# --- KiteConnect Import Guard ---
try:
    from kiteconnect import KiteConnect
    KITE_AVAILABLE = True
except ImportError:
    KiteConnect = None
    KITE_AVAILABLE = False

# --- Colab Userdata Import Guard ---
try:
    from google.colab import userdata
except ImportError:
    userdata = None

def get_secret(key, default=None):
    """Retrieve secret key from Google Colab userdata or environment variables."""
    if userdata is not None:
        try:
            val = userdata.get(key)
            if val is not None:
                return val
        except Exception:
            pass
    return os.environ.get(key, default)

# --- Transaction Type & Exchange Constants (Prevents AttributeError in Paper Trading Mode) ---
TRANSACTION_TYPE_BUY = "BUY"
TRANSACTION_TYPE_SELL = "SELL"
EXCHANGE_NFO = "NFO"
PRODUCT_MIS = "MIS"
PRODUCT_NRML = "NRML"
ORDER_TYPE_MARKET = "MARKET"
ORDER_TYPE_LIMIT = "LIMIT"

# --- General Configuration ---
PRODUCT_TYPE = PRODUCT_MIS            # MIS (Intraday), NRML (Carry Forward)
ORDER_TYPE = ORDER_TYPE_MARKET        # MARKET, LIMIT, SL, SL-M
QUANTITY_PER_LOT = 50                  # Nifty lot size
NIFTY_FUTURES_INSTRUMENT_TOKEN = 256265 # Nifty 50 futures instrument token

# --- Strategy Parameters ---
FIRST_CANDLE_START_TIME = datetime.time(9, 15) # Strategy start time
FIRST_CANDLE_END_TIME = datetime.time(9, 18)   # End of first 3-min candle
PROFIT_ENTRY_THRESHOLD_PERCENT = 0.07          # Nifty spot change to trigger trade (+0.07% or -0.07%)
OPTION_STRIKE_DIFFERENCE = 50                  # Difference for OTM/ITM strikes (e.g., 50 points)
MARKET_CLOSE_TIME = datetime.time(15, 15)      # Auto square-off time

# --- Dynamic Profit Lock (Trailing Profit Lock) ---
DYNAMIC_PROFIT_LOCK_ENABLED = True             # Enable/Disable dynamic profit lock
INITIAL_PROFIT_LOCK_TRIGGER = 600              # Initial P&L (₹) to activate profit lock
PROFIT_LOCK_BUFFER = 100                       # Buffer (₹) between current P&L and locked P&L

# --- Risk Management ---
DAILY_MAX_LOSS = 1000                          # Daily maximum loss (₹). If reached, stop trading.

# --- Trading Modes ---
PAPER_TRADING_MODE = True                      # Set to True for paper trading, False for live trading

# --- Logging Configuration ---
LOG_FILE = "nifty_algo_trade.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

logger.info("Configuration loaded successfully. Paper Trading Mode: %s", PAPER_TRADING_MODE)

# --- Global Kite Object ---
kite = None

def initialize_kite_api(api_key, api_secret, request_token=None):
    """Initialize KiteConnect session."""
    if not KITE_AVAILABLE:
        logger.error("KiteConnect module is not installed. Install via `pip install kiteconnect`.")
        return None
    try:
        k_obj = KiteConnect(api_key=api_key)
        if request_token:
            data = k_obj.generate_session(request_token, api_secret=api_secret)
            k_obj.set_access_token(data["access_token"])
            logger.info("Kite Connect session generated and access token set.")
        else:
            logger.warning("No request token provided. Generating manual login URL.")
            logger.info("Manual login URL: %s", k_obj.login_url())
        return k_obj
    except Exception as e:
        logger.error(f"Error initializing Kite Connect: {e}")
        return None

def reconnect_kite(api_key, api_secret, request_token):
    """Attempt to reconnect Kite Connect session."""
    global kite
    logger.info("Attempting to reconnect Kite...")
    kite = initialize_kite_api(api_key, api_secret, request_token)
    if kite:
        logger.info("Kite reconnected successfully.")
    else:
        logger.error("Failed to reconnect Kite. Check credentials and network.")
    return kite

# --- Order Placement and Management ---
EXECUTED_ORDERS = {}
ACTIVE_POSITIONS = defaultdict(lambda: {'quantity': 0, 'entry_price': 0, 'order_ids': []})

NET_REALIZED_PNL = 0
NET_UNREALIZED_PNL = 0

def calculate_realized_pnl(order_details):
    global NET_REALIZED_PNL
    logger.info(f"Calculating realized P&L for: {order_details}")
    try:
        entry = order_details.get('entry_price', 0)
        exit_p = order_details.get('exit_price', 0)
        qty = order_details.get('quantity', 0)
        pnl = (exit_p - entry) * qty
        NET_REALIZED_PNL += pnl
        logger.info(f"Realized P&L updated: +Rs.{pnl:.2f} (Total: Rs.{NET_REALIZED_PNL:.2f})")
    except Exception as e:
        logger.error(f"Error calculating realized P&L: {e}")

def place_order(instrument_token, tradingsymbol, transaction_type, quantity, order_type=ORDER_TYPE, product=PRODUCT_TYPE, price=0, reason=""):
    global kite
    if PAPER_TRADING_MODE:
        logger.info(f"[PAPER TRADE] Placing {transaction_type} order for {tradingsymbol} quantity {quantity}. Reason: {reason}")
        order_id = f"PAPER_ORDER_{int(time.time() * 1000)}"
        avg_price = price if price > 0 else get_ltp(instrument_token)
        if avg_price is None or avg_price == 0:
            avg_price = 100.0  # Default mock price if LTP not available

        EXECUTED_ORDERS[order_id] = {
            'instrument_token': instrument_token,
            'tradingsymbol': tradingsymbol,
            'quantity': quantity,
            'type': transaction_type,
            'status': 'complete',
            'average_price': avg_price
        }
        update_active_positions(tradingsymbol, transaction_type, quantity, avg_price)
        logger.info(f"[PAPER TRADE] Order simulated successfully. Order ID: {order_id} @ Rs.{avg_price:.2f}")
        return order_id

    if not kite:
        logger.error("Kite object not initialized. Cannot place order.")
        return None

    try:
        order_params = {
            "variety": kite.VARIETY_REGULAR,
            "exchange": kite.EXCHANGE_NFO,
            "tradingsymbol": tradingsymbol,
            "transaction_type": transaction_type,
            "quantity": quantity,
            "product": product,
            "order_type": order_type
        }
        if order_type == ORDER_TYPE_LIMIT and price > 0:
            order_params["price"] = price

        order_id = kite.place_order(**order_params)
        logger.info(f"Live order placed successfully. Order ID: {order_id}")
        EXECUTED_ORDERS[order_id] = {
            'instrument_token': instrument_token,
            'tradingsymbol': tradingsymbol,
            'quantity': quantity,
            'type': transaction_type,
            'status': 'pending'
        }
        return order_id
    except Exception as e:
        logger.error(f"Error placing order for {tradingsymbol}: {e}")
        return None

def update_active_positions(tradingsymbol, transaction_type, quantity, price):
    if transaction_type == TRANSACTION_TYPE_BUY:
        current_total_value = ACTIVE_POSITIONS[tradingsymbol]['quantity'] * ACTIVE_POSITIONS[tradingsymbol]['entry_price']
        new_total_value = current_total_value + (quantity * price)
        ACTIVE_POSITIONS[tradingsymbol]['quantity'] += quantity
        if ACTIVE_POSITIONS[tradingsymbol]['quantity'] > 0:
            ACTIVE_POSITIONS[tradingsymbol]['entry_price'] = new_total_value / ACTIVE_POSITIONS[tradingsymbol]['quantity']
        else:
            ACTIVE_POSITIONS[tradingsymbol]['entry_price'] = 0
    elif transaction_type == TRANSACTION_TYPE_SELL:
        entry_price = ACTIVE_POSITIONS[tradingsymbol]['entry_price']
        ACTIVE_POSITIONS[tradingsymbol]['quantity'] -= quantity
        if ACTIVE_POSITIONS[tradingsymbol]['quantity'] == 0:
            calculate_realized_pnl({
                'tradingsymbol': tradingsymbol,
                'quantity': quantity,
                'exit_price': price,
                'entry_price': entry_price
            })
            ACTIVE_POSITIONS[tradingsymbol]['entry_price'] = 0

    logger.debug(f"Updated position for {tradingsymbol}: {ACTIVE_POSITIONS[tradingsymbol]}")

def get_ltp(instrument_token):
    global kite
    if PAPER_TRADING_MODE:
        return 24000 + (math.sin(time.time() / 10) * 100) # Mock LTP calculation

    if not kite:
        logger.error("Kite object not initialized. Cannot fetch LTP.")
        return None
    try:
        data = kite.ltp(f"NFO:{instrument_token}")
        return data[f"NFO:{instrument_token}"]['last_price']
    except Exception as e:
        logger.error(f"Error fetching LTP for {instrument_token}: {e}")
        reconnect_kite(get_secret('KITE_API_KEY'), get_secret('KITE_API_SECRET'), get_secret('KITE_REQUEST_TOKEN'))
        return None

def get_historical_data(instrument_token, from_date, to_date, interval):
    global kite
    if PAPER_TRADING_MODE:
        logger.info("[PAPER TRADE] Returning mock historical data.")
        df = pd.DataFrame({
            'date': pd.to_datetime([from_date + datetime.timedelta(minutes=3*i) for i in range(10)]),
            'open': [23900 + i for i in range(10)],
            'high': [23910 + i * 2 for i in range(10)],
            'low': [23890 - i for i in range(10)],
            'close': [23905 + i * 1.5 for i in range(10)],
            'volume': [1000 + i*100 for i in range(10)]
        })
        return df.to_dict('records')

    if not kite:
        logger.error("Kite object not initialized. Cannot fetch historical data.")
        return None
    try:
        data = kite.historical_data(instrument_token, from_date, to_date, interval, continuous=False)
        return data
    except Exception as e:
        logger.error(f"Error fetching historical data for {instrument_token}: {e}")
        reconnect_kite(get_secret('KITE_API_KEY'), get_secret('KITE_API_SECRET'), get_secret('KITE_REQUEST_TOKEN'))
        return None

def get_option_chain(nifty_spot_price, expiry_date=None):
    if expiry_date is None:
        expiry_date = get_nearest_expiry()

    nearest_atm_strike = round(nifty_spot_price / 50) * 50
    strikes = sorted(list(set([
        nearest_atm_strike - 200, nearest_atm_strike - 150, nearest_atm_strike - 100, nearest_atm_strike - 50,
        nearest_atm_strike,
        nearest_atm_strike + 50, nearest_atm_strike + 100, nearest_atm_strike + 150, nearest_atm_strike + 200
    ])))

    options_data = []
    mock_token_counter = 100000
    date_str = expiry_date.strftime('%y%b%d').upper()
    for strike in strikes:
        ce_symbol = f"NIFTY{date_str}CE{strike}"
        options_data.append({
            'instrument_token': mock_token_counter,
            'tradingsymbol': ce_symbol,
            'name': 'NIFTY',
            'expiry': expiry_date,
            'strike': strike,
            'instrument_type': 'CE',
            'segment': 'NFO'
        })
        mock_token_counter += 1

        pe_symbol = f"NIFTY{date_str}PE{strike}"
        options_data.append({
            'instrument_token': mock_token_counter,
            'tradingsymbol': pe_symbol,
            'name': 'NIFTY',
            'expiry': expiry_date,
            'strike': strike,
            'instrument_type': 'PE',
            'segment': 'NFO'
        })
        mock_token_counter += 1

    df_options = pd.DataFrame(options_data)
    logger.info(f"Generated option chain for spot {nifty_spot_price:.2f}.")
    return df_options

def get_nearest_expiry():
    today = datetime.date.today()
    days_until_thursday = (3 - today.weekday() + 7) % 7
    if days_until_thursday == 0 and today.weekday() == 3:
        if datetime.datetime.now().time() > MARKET_CLOSE_TIME:
            next_thursday = today + datetime.timedelta(days=7)
        else:
            next_thursday = today
    else:
        next_thursday = today + datetime.timedelta(days=days_until_thursday if days_until_thursday > 0 else 7)

    logger.info(f"Nearest expiry date: {next_thursday.strftime('%Y-%m-%d')}")
    return next_thursday

# --- Emergency Kill Switch ---
EMERGENCY_KILL_SWITCH_ACTIVE = False

def activate_kill_switch():
    global EMERGENCY_KILL_SWITCH_ACTIVE
    EMERGENCY_KILL_SWITCH_ACTIVE = True
    logger.warning("EMERGENCY KILL SWITCH ACTIVATED! All positions will be squared off.")

def deactivate_kill_switch():
    global EMERGENCY_KILL_SWITCH_ACTIVE
    EMERGENCY_KILL_SWITCH_ACTIVE = False
    logger.info("EMERGENCY KILL SWITCH DEACTIVATED.")

# --- Strategy State Variables ---
first_candle_high = 0
first_candle_low = 0
first_candle_close = 0
initial_nifty_spot_price = 0

TRADE_INITIATED = False
SPREAD_TYPE = None

ACTIVE_SPREAD_POSITIONS = {
    'buy_leg': None,
    'sell_leg': None
}

CURRENT_LOCKED_PROFIT = 0

def get_nifty_spot_price():
    if PAPER_TRADING_MODE:
        mock_spot = 23925 + (math.sin(time.time() / 60) * 50)
        logger.info(f"[PAPER TRADE] Mock Nifty Spot Price: {mock_spot:.2f}")
        return mock_spot

    spot_ltp = get_ltp(NIFTY_FUTURES_INSTRUMENT_TOKEN)
    if spot_ltp:
        logger.info(f"Nifty Futures LTP: {spot_ltp:.2f}")
        return spot_ltp
    logger.error("Could not fetch Nifty spot price.")
    return None

def get_atm_strike(spot_price):
    return round(spot_price / 50) * 50

def select_strikes(nifty_spot_price, option_type):
    atm_strike = get_atm_strike(nifty_spot_price)
    nearest_expiry = get_nearest_expiry()
    option_chain_df = get_option_chain(nifty_spot_price, nearest_expiry)

    if option_type == 'PE':
        buy_strike = atm_strike - OPTION_STRIKE_DIFFERENCE
        sell_strike = atm_strike + OPTION_STRIKE_DIFFERENCE
    else:
        buy_strike = atm_strike + OPTION_STRIKE_DIFFERENCE
        sell_strike = atm_strike - OPTION_STRIKE_DIFFERENCE

    buy_df = option_chain_df[(option_chain_df['strike'] == buy_strike) & (option_chain_df['instrument_type'] == option_type)]
    sell_df = option_chain_df[(option_chain_df['strike'] == sell_strike) & (option_chain_df['instrument_type'] == option_type)]

    buy_option = buy_df.iloc[0] if not buy_df.empty else None
    sell_option = sell_df.iloc[0] if not sell_df.empty else None

    if buy_option is None or sell_option is None:
        logger.error(f"Could not find options for selected strikes: Buy {buy_strike} {option_type}, Sell {sell_strike} {option_type}")
        return None, None

    logger.info(f"Selected strikes: Buy {buy_option['tradingsymbol']}, Sell {sell_option['tradingsymbol']}")
    return buy_option, sell_option

def enter_spread(spread_type, nifty_spot_price):
    global TRADE_INITIATED, SPREAD_TYPE, ACTIVE_SPREAD_POSITIONS
    if TRADE_INITIATED:
        logger.warning("Trade already initiated. Cannot enter new spread.")
        return False

    option_type = 'PE' if spread_type == 'PUT_SPREAD' else 'CE'
    buy_option, sell_option = select_strikes(nifty_spot_price, option_type)

    if buy_option is None or sell_option is None:
        logger.error("Failed to select appropriate strikes. Aborting spread entry.")
        return False

    buy_order_id = place_order(buy_option['instrument_token'], buy_option['tradingsymbol'], TRANSACTION_TYPE_BUY, QUANTITY_PER_LOT, reason="Buy Leg Entry")
    time.sleep(0.5)
    sell_order_id = place_order(sell_option['instrument_token'], sell_option['tradingsymbol'], TRANSACTION_TYPE_SELL, QUANTITY_PER_LOT, reason="Sell Leg Entry")

    if buy_order_id and sell_order_id:
        ACTIVE_SPREAD_POSITIONS['buy_leg'] = {
            'instrument_token': buy_option['instrument_token'],
            'tradingsymbol': buy_option['tradingsymbol'],
            'strike': buy_option['strike'],
            'type': option_type,
            'entry_price': EXECUTED_ORDERS[buy_order_id].get('average_price'),
            'quantity': QUANTITY_PER_LOT,
            'order_id': buy_order_id
        }
        ACTIVE_SPREAD_POSITIONS['sell_leg'] = {
            'instrument_token': sell_option['instrument_token'],
            'tradingsymbol': sell_option['tradingsymbol'],
            'strike': sell_option['strike'],
            'type': option_type,
            'entry_price': EXECUTED_ORDERS[sell_order_id].get('average_price'),
            'quantity': QUANTITY_PER_LOT,
            'order_id': sell_order_id,
            're_entry_mark': {'low': 0, 'high': 0, 'candle_type': None}
        }

        if spread_type == 'PUT_SPREAD':
            ACTIVE_SPREAD_POSITIONS['sell_leg']['stop_loss_price'] = first_candle_low
            logger.info(f"Put Spread entered. Sell Leg SL set to first_candle_low: {first_candle_low}")
        else:
            ACTIVE_SPREAD_POSITIONS['sell_leg']['stop_loss_price'] = first_candle_high
            logger.info(f"Call Spread entered. Sell Leg SL set to first_candle_high: {first_candle_high}")

        TRADE_INITIATED = True
        SPREAD_TYPE = spread_type
        logger.info(f"Successfully entered {spread_type}.")
        return True
    else:
        logger.error(f"Failed to enter {spread_type}.")
        return False

def get_net_pnl():
    global NET_REALIZED_PNL, NET_UNREALIZED_PNL
    current_pnl = NET_REALIZED_PNL

    if ACTIVE_SPREAD_POSITIONS['buy_leg']:
        leg = ACTIVE_SPREAD_POSITIONS['buy_leg']
        ltp = get_ltp(leg['instrument_token'])
        if ltp is not None and leg.get('entry_price'):
            current_pnl += (ltp - leg['entry_price']) * leg['quantity']

    if ACTIVE_SPREAD_POSITIONS['sell_leg'] and ACTIVE_SPREAD_POSITIONS['sell_leg'].get('quantity', 0) > 0:
        leg = ACTIVE_SPREAD_POSITIONS['sell_leg']
        ltp = get_ltp(leg['instrument_token'])
        if ltp is not None and leg.get('entry_price'):
            current_pnl += (leg['entry_price'] - ltp) * leg['quantity']

    NET_UNREALIZED_PNL = current_pnl - NET_REALIZED_PNL
    logger.debug(f"Current Net P&L: {current_pnl:.2f}")
    return current_pnl

def update_dynamic_profit_lock(current_pnl):
    global CURRENT_LOCKED_PROFIT
    if not DYNAMIC_PROFIT_LOCK_ENABLED:
        return

    if current_pnl >= INITIAL_PROFIT_LOCK_TRIGGER:
        new_locked_profit = current_pnl - PROFIT_LOCK_BUFFER
        if new_locked_profit > CURRENT_LOCKED_PROFIT:
            CURRENT_LOCKED_PROFIT = new_locked_profit
            logger.info(f"Dynamic Profit Lock updated. New locked_profit: {CURRENT_LOCKED_PROFIT:.2f}")

def square_off_all_positions(reason="Bot triggered square-off"):
    global TRADE_INITIATED, ACTIVE_SPREAD_POSITIONS
    logger.warning(f"Squaring off all active positions. Reason: {reason}")

    if ACTIVE_SPREAD_POSITIONS['buy_leg']:
        leg = ACTIVE_SPREAD_POSITIONS['buy_leg']
        logger.info(f"Squaring off buy leg: {leg['tradingsymbol']}")
        place_order(leg['instrument_token'], leg['tradingsymbol'], TRANSACTION_TYPE_SELL, leg['quantity'], reason=f"Squareoff: {reason}")
        ACTIVE_SPREAD_POSITIONS['buy_leg'] = None

    if ACTIVE_SPREAD_POSITIONS['sell_leg'] and ACTIVE_SPREAD_POSITIONS['sell_leg'].get('quantity', 0) > 0:
        leg = ACTIVE_SPREAD_POSITIONS['sell_leg']
        logger.info(f"Squaring off sell leg: {leg['tradingsymbol']}")
        place_order(leg['instrument_token'], leg['tradingsymbol'], TRANSACTION_TYPE_BUY, leg['quantity'], reason=f"Squareoff: {reason}")
        ACTIVE_SPREAD_POSITIONS['sell_leg'] = None

    TRADE_INITIATED = False
    logger.info("All positions squared off.")

def check_daily_max_loss():
    current_net_pnl = get_net_pnl()
    if current_net_pnl < -DAILY_MAX_LOSS:
        logger.critical(f"DAILY MAX LOSS REACHED! Current P&L: {current_net_pnl:.2f}. Limit: {-DAILY_MAX_LOSS:.2f}")
        square_off_all_positions(reason="Daily Max Loss reached.")
        return True
    return False

def check_and_handle_sell_leg_sl(current_nifty_close, current_candle_high, current_candle_low):
    global ACTIVE_SPREAD_POSITIONS
    sell_leg = ACTIVE_SPREAD_POSITIONS['sell_leg']
    if not sell_leg or 'stop_loss_price' not in sell_leg or sell_leg.get('stop_loss_price') is None:
        return False

    sl_triggered = False
    if SPREAD_TYPE == 'CALL_SPREAD':
        if current_nifty_close > sell_leg['stop_loss_price']:
            logger.warning(f"Call Spread Sell Leg SL Triggered! Nifty Close ({current_nifty_close:.2f}) > SL Price ({sell_leg['stop_loss_price']:.2f})")
            sl_triggered = True
    elif SPREAD_TYPE == 'PUT_SPREAD':
        if current_nifty_close < sell_leg['stop_loss_price']:
            logger.warning(f"Put Spread Sell Leg SL Triggered! Nifty Close ({current_nifty_close:.2f}) < SL Price ({sell_leg['stop_loss_price']:.2f})")
            sl_triggered = True

    if sl_triggered:
        leg = ACTIVE_SPREAD_POSITIONS['sell_leg']
        logger.info(f"Squaring off SL triggered sell leg: {leg['tradingsymbol']}")
        place_order(leg['instrument_token'], leg['tradingsymbol'], TRANSACTION_TYPE_BUY, leg['quantity'], reason="Sell Leg SL Hit")
        ACTIVE_SPREAD_POSITIONS['sell_leg'] = {'instrument_token': None, 'tradingsymbol': None, 'strike': None, 'type': None, 'entry_price': None, 'quantity': 0, 'stop_loss_price': None, 're_entry_mark': {'low': 0, 'high': 0, 'candle_type': None}}
        return True
    return False

def attempt_sell_leg_reentry(current_nifty_close, current_candle_open, current_candle_high, current_candle_low):
    global ACTIVE_SPREAD_POSITIONS
    sell_leg_state = ACTIVE_SPREAD_POSITIONS['sell_leg']
    if not sell_leg_state or sell_leg_state.get('quantity', 0) > 0:
        return False

    re_entry_mark = sell_leg_state['re_entry_mark']

    if SPREAD_TYPE == 'CALL_SPREAD':
        if re_entry_mark['candle_type'] is None:
            if current_nifty_close < current_candle_open:
                re_entry_mark['low'] = current_candle_low
                re_entry_mark['high'] = current_candle_high
                re_entry_mark['candle_type'] = 'RED'
                logger.info(f"[CALL SPREAD RE-ENTRY] Marked first red candle: Low={current_candle_low}, High={current_candle_high}")
        elif re_entry_mark['candle_type'] == 'RED':
            if current_nifty_close < re_entry_mark['low']:
                logger.info(f"[CALL SPREAD RE-ENTRY] Re-entry confirmed! Nifty Close ({current_nifty_close:.2f}) < Red Candle Low ({re_entry_mark['low']:.2f})")
                option_type = 'CE'
                _, sell_option_for_reentry = select_strikes(current_nifty_close, option_type)
                if sell_option_for_reentry is not None:
                    re_entry_order_id = place_order(sell_option_for_reentry['instrument_token'], sell_option_for_reentry['tradingsymbol'], TRANSACTION_TYPE_SELL, QUANTITY_PER_LOT, reason="Call Spread Re-entry")
                    if re_entry_order_id:
                        ACTIVE_SPREAD_POSITIONS['sell_leg'] = {
                            'instrument_token': sell_option_for_reentry['instrument_token'],
                            'tradingsymbol': sell_option_for_reentry['tradingsymbol'],
                            'strike': sell_option_for_reentry['strike'],
                            'type': option_type,
                            'entry_price': EXECUTED_ORDERS[re_entry_order_id].get('average_price'),
                            'quantity': QUANTITY_PER_LOT,
                            'order_id': re_entry_order_id,
                            'stop_loss_price': re_entry_mark['high'],
                            're_entry_mark': {'low': 0, 'high': 0, 'candle_type': None}
                        }
                        logger.info(f"[CALL SPREAD RE-ENTRY] Re-entered Call Spread Sell Leg. New SL: {re_entry_mark['high']:.2f}")
                        return True
                re_entry_mark['candle_type'] = None
            elif current_nifty_close > re_entry_mark['high']:
                logger.info(f"[CALL SPREAD RE-ENTRY] Discarded re-entry trigger. Nifty Close ({current_nifty_close:.2f}) > Red Candle High ({re_entry_mark['high']:.2f})")
                re_entry_mark['candle_type'] = None

    elif SPREAD_TYPE == 'PUT_SPREAD':
        if re_entry_mark['candle_type'] is None:
            if current_nifty_close > current_candle_open:
                re_entry_mark['low'] = current_candle_low
                re_entry_mark['high'] = current_candle_high
                re_entry_mark['candle_type'] = 'GREEN'
                logger.info(f"[PUT SPREAD RE-ENTRY] Marked first green candle: Low={current_candle_low}, High={current_candle_high}")
        elif re_entry_mark['candle_type'] == 'GREEN':
            if current_nifty_close > re_entry_mark['high']:
                logger.info(f"[PUT SPREAD RE-ENTRY] Re-entry confirmed! Nifty Close ({current_nifty_close:.2f}) > Green Candle High ({re_entry_mark['high']:.2f})")
                option_type = 'PE'
                _, sell_option_for_reentry = select_strikes(current_nifty_close, option_type)
                if sell_option_for_reentry is not None:
                    re_entry_order_id = place_order(sell_option_for_reentry['instrument_token'], sell_option_for_reentry['tradingsymbol'], TRANSACTION_TYPE_SELL, QUANTITY_PER_LOT, reason="Put Spread Re-entry")
                    if re_entry_order_id:
                        ACTIVE_SPREAD_POSITIONS['sell_leg'] = {
                            'instrument_token': sell_option_for_reentry['instrument_token'],
                            'tradingsymbol': sell_option_for_reentry['tradingsymbol'],
                            'strike': sell_option_for_reentry['strike'],
                            'type': option_type,
                            'entry_price': EXECUTED_ORDERS[re_entry_order_id].get('average_price'),
                            'quantity': QUANTITY_PER_LOT,
                            'order_id': re_entry_order_id,
                            'stop_loss_price': re_entry_mark['low'],
                            're_entry_mark': {'low': 0, 'high': 0, 'candle_type': None}
                        }
                        logger.info(f"[PUT SPREAD RE-ENTRY] Re-entered Put Spread Sell Leg. New SL: {re_entry_mark['low']:.2f}")
                        return True
                re_entry_mark['candle_type'] = None
            elif current_nifty_close < re_entry_mark['low']:
                logger.info(f"[PUT SPREAD RE-ENTRY] Discarded re-entry trigger. Nifty Close ({current_nifty_close:.2f}) < Green Candle Low ({re_entry_mark['low']:.2f})")
                re_entry_mark['candle_type'] = None

    return False

def main_strategy_loop(max_ticks=None):
    global first_candle_high, first_candle_low, first_candle_close, initial_nifty_spot_price, TRADE_INITIATED, CURRENT_LOCKED_PROFIT, EMERGENCY_KILL_SWITCH_ACTIVE, kite

    logger.info("Starting Nifty 50 Options Trading Bot.")

    if not PAPER_TRADING_MODE:
        KITE_API_KEY = get_secret('KITE_API_KEY')
        KITE_API_SECRET = get_secret('KITE_API_SECRET')
        KITE_REQUEST_TOKEN = get_secret('KITE_REQUEST_TOKEN')

        if not KITE_API_KEY or not KITE_REQUEST_TOKEN:
            logger.error("KITE_API_KEY or KITE_REQUEST_TOKEN not found in environment/secrets.")
            if KITE_AVAILABLE and KITE_API_KEY:
                print(f"Login URL: {KiteConnect(api_key=KITE_API_KEY).login_url()}")
            return

        kite = initialize_kite_api(KITE_API_KEY, KITE_API_SECRET, KITE_REQUEST_TOKEN)
        if not kite:
            logger.critical("Failed to initialize Kite API. Exiting bot.")
            return
    else:
        logger.info("Running in PAPER TRADING MODE. Mocking Kite API calls.")

    tick_counter = 0

    while True:
        tick_counter += 1
        if max_ticks and tick_counter > max_ticks:
            logger.info("Reached maximum ticks limit (%d). Stopping loop for demonstration.", max_ticks)
            break

        current_time = datetime.datetime.now().time()
        today_date = datetime.date.today()
        logger.debug(f"Current time: {current_time}")

        if EMERGENCY_KILL_SWITCH_ACTIVE:
            square_off_all_positions(reason="Emergency Kill Switch Activated")
            logger.critical("Emergency Kill Switch is active. Stopping bot.")
            break

        if check_daily_max_loss():
            logger.critical("Daily max loss reached. Stopping strategy for the day.")
            break

        # --- Phase 1: Determine First 3-min Candle Close ---
        # Fixed logic flaw: Changed nested condition to execute correctly at or after FIRST_CANDLE_END_TIME
        if not first_candle_close:
            if PAPER_TRADING_MODE or current_time >= FIRST_CANDLE_END_TIME:
                try:
                    from_ts = datetime.datetime.combine(today_date, FIRST_CANDLE_START_TIME)
                    to_ts = datetime.datetime.combine(today_date, FIRST_CANDLE_END_TIME)
                    historical_data = get_historical_data(NIFTY_FUTURES_INSTRUMENT_TOKEN, from_ts, to_ts, "3minute")

                    if historical_data and len(historical_data) > 0:
                        first_candle_data = historical_data[-1]
                        first_candle_high = first_candle_data['high']
                        first_candle_low = first_candle_data['low']
                        first_candle_close = first_candle_data['close']
                        initial_nifty_spot_price = historical_data[0]['open']
                        logger.info(f"First 3-min candle (9:15-9:18) captured: High={first_candle_high:.2f}, Low={first_candle_low:.2f}, Close={first_candle_close:.2f}")
                    else:
                        logger.warning("No historical data for first 3-min candle yet.")
                except Exception as e:
                    logger.error(f"Error fetching first 3-min candle data: {e}")

        # --- Phase 2: Entry Conditions ---
        if first_candle_close and not TRADE_INITIATED:
            current_nifty_spot = get_nifty_spot_price()
            if current_nifty_spot is not None and initial_nifty_spot_price > 0:
                percentage_change = ((current_nifty_spot - initial_nifty_spot_price) / initial_nifty_spot_price) * 100
                logger.info(f"Current Nifty Spot: {current_nifty_spot:.2f}, Initial Spot: {initial_nifty_spot_price:.2f}, Change: {percentage_change:.2f}%")

                if percentage_change > PROFIT_ENTRY_THRESHOLD_PERCENT:
                    logger.info(f"Nifty Change (+{percentage_change:.2f}%) > +{PROFIT_ENTRY_THRESHOLD_PERCENT}%. Entering Put Spread.")
                    enter_spread('PUT_SPREAD', current_nifty_spot)
                elif percentage_change < -PROFIT_ENTRY_THRESHOLD_PERCENT:
                    logger.info(f"Nifty Change ({percentage_change:.2f}%) < -{PROFIT_ENTRY_THRESHOLD_PERCENT}%. Entering Call Spread.")
                    enter_spread('CALL_SPREAD', current_nifty_spot)

        # --- Phase 3: Manage Active Trades ---
        if TRADE_INITIATED:
            current_nifty_spot = get_nifty_spot_price()
            if current_nifty_spot is not None:
                try:
                    history_to_time = datetime.datetime.now()
                    history_from_time = history_to_time - datetime.timedelta(minutes=5)
                    recent_candles = get_historical_data(NIFTY_FUTURES_INSTRUMENT_TOKEN, history_from_time, history_to_time, "3minute")
                    if recent_candles and len(recent_candles) > 0:
                        last_candle_data = recent_candles[-1]
                        current_candle_high = last_candle_data['high']
                        current_candle_low = last_candle_data['low']
                        current_candle_open = last_candle_data['open']
                        current_nifty_close_for_sl = last_candle_data['close']

                        if ACTIVE_SPREAD_POSITIONS['sell_leg'] and ACTIVE_SPREAD_POSITIONS['sell_leg'].get('quantity', 0) > 0:
                            check_and_handle_sell_leg_sl(current_nifty_close_for_sl, current_candle_high, current_candle_low)
                        else:
                            attempt_sell_leg_reentry(current_nifty_close_for_sl, current_candle_open, current_candle_high, current_candle_low)
                except Exception as e:
                    logger.error(f"Error checking SL/Re-entry: {e}")

                current_net_pnl = get_net_pnl()
                update_dynamic_profit_lock(current_net_pnl)

                if DYNAMIC_PROFIT_LOCK_ENABLED and CURRENT_LOCKED_PROFIT > 0:
                    if current_net_pnl <= CURRENT_LOCKED_PROFIT:
                        logger.warning(f"Dynamic Profit Lock Triggered! Current P&L ({current_net_pnl:.2f}) <= Locked Profit ({CURRENT_LOCKED_PROFIT:.2f})")
                        square_off_all_positions(reason="Dynamic Profit Lock triggered.")
                        CURRENT_LOCKED_PROFIT = 0

        # --- Phase 4: Auto Square-off at Market Close ---
        if current_time >= MARKET_CLOSE_TIME and TRADE_INITIATED:
            logger.info(f"Market close time ({MARKET_CLOSE_TIME}) reached. Squaring off all positions.")
            square_off_all_positions(reason="Market close auto square-off.")
            break

        if max_ticks:
            time.sleep(0.5)
        else:
            time.sleep(10)

    logger.info("Nifty 50 Options Trading Bot loop finished.")

def render_dashboard():
    """Render HTML dashboard for Jupyter notebooks or summary log for terminal."""
    try:
        import IPython.display as display
        html_output = f"""
        <div style="font-family: Arial, sans-serif; border: 1px solid #ccc; padding: 15px; border-radius: 8px; background-color: #1e1e2e; color: #cdd6f4;">
            <h2 style="color: #89b4fa; margin-top: 0;">AlgoBeat Trading Dashboard</h2>
            <p>Last updated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
            <div style="display: flex; justify-content: space-around; margin-bottom: 20px; border-bottom: 1px solid #45475a; padding-bottom: 15px;">
                <div style="text-align: center;">
                    <h3 style="color: #a6e3a1;">P&L Overview</h3>
                    <p>Realized P&L: <strong>₹{NET_REALIZED_PNL:.2f}</strong></p>
                    <p>Unrealized P&L: <strong>₹{NET_UNREALIZED_PNL:.2f}</strong></p>
                    <p><strong>Total P&L: ₹{(NET_REALIZED_PNL + NET_UNREALIZED_PNL):.2f}</strong></p>
                </div>
                <div style="text-align: center;">
                    <h3 style="color: #fab387;">Risk Management</h3>
                    <p>Daily Max Loss: <strong>₹{DAILY_MAX_LOSS:.2f}</strong></p>
                    <p>Kill Switch: <strong>{'ACTIVE' if EMERGENCY_KILL_SWITCH_ACTIVE else 'INACTIVE'}</strong></p>
                    <p>Dynamic Profit Lock: <strong>{'ENABLED' if DYNAMIC_PROFIT_LOCK_ENABLED else 'DISABLED'}</strong> (Locked: ₹{CURRENT_LOCKED_PROFIT:.2f})</p>
                </div>
                <div style="text-align: center;">
                    <h3 style="color: #cba6f7;">Strategy State</h3>
                    <p>Trading Mode: <strong>{'PAPER' if PAPER_TRADING_MODE else 'LIVE'}</strong></p>
                    <p>Trade Initiated: <strong>{'YES' if TRADE_INITIATED else 'NO'}</strong></p>
                    <p>Spread Type: <strong>{SPREAD_TYPE if SPREAD_TYPE else 'N/A'}</strong></p>
                </div>
            </div>
        </div>
        """
        display.clear_output(wait=True)
        display.display(display.HTML(html_output))
    except Exception:
        logger.info(f"[DASHBOARD] Realized P&L: ₹{NET_REALIZED_PNL:.2f} | Unrealized P&L: ₹{NET_UNREALIZED_PNL:.2f} | Mode: {'PAPER' if PAPER_TRADING_MODE else 'LIVE'}")

if __name__ == "__main__":
    logger.info("Running Nifty 50 Options Trading Bot in standalone mode...")
    main_strategy_loop(max_ticks=5)
